In [ ]:
from manim import *
import numpy as np
from scipy.spatial.transform import Rotation as R

# Intrinsic coordinates are U-V plane coordinates,
# Extrinsic coordinates are local coordinates (i.e. x-y-z)


def param_uvplane_rotated(x, y, ang_x=0, ang_y=0):
    # Parametrize a plane rotated about the u and v axes
    z = np.tan(ang_y) * x + np.tan(ang_x) * y
    return np.array([x, y, z])


def uv_normal_unit_vector(ang_x, ang_y):
    x = np.sin(ang_y)
    y = np.sin(ang_x)
    z = np.sqrt(1 - x**2 - y**2)
    return np.array([x, y, z])


class ThreeDuvCoordinatePlot(ThreeDScene):
    def construct(self):
        resolution_fa = 24
        self.set_camera_orientation(phi=75 * DEGREES, theta=0 * DEGREES)

        # Angle of rotation (i.e. angle between zenith and tracking center)
        ang_x = np.deg2rad(-40)
        ang_y = np.deg2rad(0)

        def param_uvplane(u, v):
            return param_uvplane_rotated(u, v, ang_x, ang_y)

        def uv_normal_unit_vector():
            x = np.sin(-ang_y)
            y = np.sin(-ang_x)
            z = np.sqrt(1 - x**2 - y**2)
            return np.array([x, y, z])

        # Initialize u-v surface
        uv_scale = 2
        uv_len = 2
        uv_range = [-uv_len, +uv_len]
        uv_plane = Surface(
            param_uvplane,
            resolution=(resolution_fa, resolution_fa),
            v_range=uv_range,
            u_range=uv_range,
        )
        uv_plane.scale(uv_scale, about_point=ORIGIN)
        uv_plane.set_style(fill_opacity=1, stroke_color=GREEN)
        uv_plane.set_fill_by_checkerboard(ORANGE, BLUE, opacity=0.2)

        # Manually initialize u-v axes as a set of arrows
        def make_uv_label(text, pos, dir=np.array([0, 0, 1])):
            return Text(text, color=BLUE, background_stroke_color=WHITE).next_to(
                pos, dir
            ).rotate(PI / 2, axis=np.array([1, 0, 0]))

        uv_axes = VGroup()
        uv_axes.add(  # u-axis
            Arrow3D(
                start=param_uvplane(-uv_len * uv_scale, 0),
                end=(u_axis_tip := param_uvplane(+uv_len * uv_scale, 0)),
                color=BLUE,
            )
        )
        uv_axes.add(  # u label
            make_uv_label("U", u_axis_tip)
        )
        uv_axes.add(  # v-axis
            Arrow3D(
                start=param_uvplane(0, -uv_len * uv_scale),
                end=(v_axis_tip := param_uvplane(0, +uv_len * uv_scale)),
                color=BLUE,
            )
        )
        uv_axes.add(  # v label
            make_uv_label("V", v_axis_tip)
        )
        e_z = uv_normal_unit_vector()
        uv_axes.add(  # w-axis
            Arrow3D(
                start=-e_z * uv_len * uv_scale,
                end=(w_axis_tip := e_z * uv_len * uv_scale), color=BLUE,
            )
        )
        uv_axes.add(  # w label
            make_uv_label("W", w_axis_tip, dir=np.array([0, 0, -1]))
        )

        # Initialize local axes
        axes = ThreeDAxes()
        local_labels = axes.get_axis_labels(
            Text("East"), Text("North"), Text("Zenith")
        )
        self.add(axes, local_labels)

        # Add elements to scene
        # self.add(axes, uv_plane, uv_axes)
        self.add(axes, uv_plane, uv_axes)

        # Camera rotation (actual animation part)
        duration = 10 # seconds
        self.begin_ambient_camera_rotation(rate=-2*PI/duration, about="theta")
        self.wait(duration=duration)

In [ ]:
%manim -qh ThreeDuvCoordinatePlot